In [ ]:
%matplotlib inline

import _plotting as plot
import jax
from jax import numpy as jnp
from matplotlib import pyplot as plt

from xxm.core.align import align_affine as align_latent
from xxm.lds import GaussianLDS

In [ ]:
def make_rotating_lds(
    angle: float = 0.15,
    process_noise: float = 0.01,
    observation_noise: float = 0.1,
) -> GaussianLDS:
    rotation = jnp.array(
        [
            [jnp.cos(angle), -jnp.sin(angle)],
            [jnp.sin(angle), jnp.cos(angle)],
        ]
    )

    return GaussianLDS.from_params(
        initial_mean=jnp.array([1.0, 0.0]),
        initial_covariance=0.01 * jnp.eye(2),
        dynamics_coefficients=rotation,
        dynamics_bias=jnp.zeros(2),
        dynamics_covariance=process_noise * jnp.eye(2),
        emission_coefficients=jnp.array(
            [
                [1.0, 0.5],
                [-0.3, 0.8],
            ]
        ),
        emission_bias=jnp.zeros(2),
        emission_covariance=observation_noise * jnp.eye(2),
    )


true_model = make_rotating_lds()

true_latents, observations = true_model.sample(
    key=jax.random.key(0),
    num_steps=300,
)


def plot_latent_data(latents, observations):

    _f, axs = plt.subplots(ncols=2, figsize=(6, 4), sharex=True, sharey=True)

    ax = axs[0]
    plot.plot_traces_2d(ax, latents)
    ax.set(title='latent states', aspect='equal')

    ax = axs[1]
    ax.plot(observations[:, 0], observations[:, 1], color='xkcd:magenta')
    ax.set(title='observations', aspect='equal')


plot_latent_data(true_latents, observations)

In [ ]:
posterior, _ = true_model.infer(observations)

In [ ]:
def plot_inference(states, posterior):
    _f, axs = plt.subplots(
        ncols=2, gridspec_kw={'width_ratios': [1, 2]}, figsize=(9, 3)
    )

    ax = axs[0]
    ax.plot(states[:, 0], states[:, 1], label='True', color='k')
    ax.plot(
        posterior.means[:, 0],
        posterior.means[:, 1],
        label='inferred mean',
        color='xkcd:apple green',
    )

    ax.set(
        title='latent states',
        aspect='equal',
    )

    ax.legend(loc='upper right')

    ax = axs[1]

    std = jnp.sqrt(posterior.covariances[:, 0, 0])

    ax.plot(states[:, 0], label='true', color='k')
    ax.plot(posterior.means[:, 0], label='posterior', color='xkcd:apple green')
    ax.fill_between(
        jnp.arange(len(states)),
        posterior.means[:, 0] - 2 * std,
        posterior.means[:, 0] + 2 * std,
        alpha=0.2,
    )


plot_inference(true_latents, posterior)

In [ ]:
initial_models = GaussianLDS.from_pca_many(
    observations,
    latent_dim=2,
    covariance_floors=jnp.logspace(-5, -2, 64),
)


fits = GaussianLDS.fit_many(
    initial_models,
    observations,
    num_iters=100,
)


best_idx = fits.best_index()
initial_model = initial_models[best_idx]
fitted_model = fits.get(best_idx)[0]

plot.plot_fit_progress_many(fits, highlight_idx=best_idx)

In [ ]:
def align_model(learned_model, true_latents):

    posterior, _ = learned_model.infer(observations)

    inferred_latents = learned_model.latent_mean(posterior)

    alignment = align_latent(inferred_latents, true_latents)

    return learned_model.align(alignment)


initial_model = align_model(initial_model, true_latents)
fitted_model = align_model(fitted_model, true_latents)

initial_posterior, _ = initial_model.infer(observations)
initial_latent = initial_model.latent_mean(initial_posterior)

fitted_posterior, _ = fitted_model.infer(observations)
fitted_latent = fitted_model.latent_mean(fitted_posterior)

In [ ]:
def plot_latent_comparison(latents_true, initial_latent, fitted_latent):
    _, axes = plt.subplots(1, 3, figsize=(12, 4), sharex='all', sharey='all')

    ax = axes[0]
    plot.plot_traces_2d(ax, latents_true)
    ax.set(
        title='True latent',
        aspect='equal',
    )

    ax = axes[1]
    plot.plot_traces_2d(ax, initial_latent, color='xkcd:grey')
    ax.set(
        title='Initial posterior',
        aspect='equal',
    )

    ax = axes[2]
    plot.plot_traces_2d(ax, fitted_latent, color='xkcd:apple green')
    ax.set(
        title='Fitted posterior',
        aspect='equal',
    )


plot_latent_comparison(true_latents, initial_latent, fitted_latent)

In [ ]:
def plot_dynamics(true_model, initial_model, fitted_model):
    _, axs = plt.subplots(
        ncols=2,
        sharex='all',
        sharey='all',
        figsize=(8, 4),
    )

    plot.plot_dyn_linear_gaussian_comparison(
        true_model.dynamics,
        initial_model.dynamics,
        color1='xkcd:grey',
        ax=axs[0],
    )
    plot.plot_dyn_linear_gaussian_comparison(
        true_model.dynamics,
        fitted_model.dynamics,
        color1='xkcd:apple green',
        ax=axs[1],
    )


plot_dynamics(true_model, initial_model, fitted_model)

In [ ]:
initial_prediction = initial_model.observation_mean(initial_posterior)

fitted_prediction = fitted_model.observation_mean(fitted_posterior)

In [ ]:
def plot_observations(observations, initial_prediction, fitted_prediction):
    _f, ax = plt.subplots()

    ax.plot(observations[:, 0], alpha=0.4, label='observed', color='k')
    ax.plot(initial_prediction[:, 0], label='initial', color='xkcd:apple green')
    ax.plot(fitted_prediction[:, 0], label='fitted', color='xkcd:coral')
    ax.legend()


plot_observations(observations, initial_prediction, fitted_prediction)